In [1]:
!pip install -q langchain-groq
%pip install -q langchain-community
%pip install -q langchain-classic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [29]:
import os, requests
from google.colab import userdata
from langchain_groq import ChatGroq
from pathlib import Path
from langchain_core.prompts import (ChatPromptTemplate,
                                    HumanMessagePromptTemplate,
                                    SystemMessagePromptTemplate,
                                    AIMessagePromptTemplate,
                                    MessagesPlaceholder
                                    )
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_classic.chains.llm import LLMChain
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory, CombinedMemory
from langchain_classic.globals import set_verbose


set_verbose(True)

In [3]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [4]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    model_kwargs= {"seed" : 350 },
    temperature= 0,
    max_tokens= 250
)

chatmessage history

In [6]:
background_info = ChatMessageHistory()
background_info.add_user_message('whats the biggest planet of our solar system ?')
background_info.add_ai_message('Jupiter. It’s more than twice as massive as all the other planets in the solar system combined')

/tmp/ipykernel_3493/4136491233.py:1: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  background_info = ChatMessageHistory()


In [7]:
human_template = HumanMessagePromptTemplate.from_template('{question}')
system_template = SystemMessagePromptTemplate.from_template("you are a sarcastic chatbot that answers in sarcastic way")

chat_template = ChatPromptTemplate.from_messages( background_info.messages + [human_template])

print(chat_template.messages)

[HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}), AIMessage(content='Jupiter. It’s more than twice as massive as all the other planets in the solar system combined', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [8]:
chain = LLMChain(
    llm = llm,
    prompt= chat_template
)

/tmp/ipykernel_3493/2382284095.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


In [9]:
response =  chain.invoke({'question' : 'how wide it is than earth ?'})

print(response['text'])



> Entering new LLMChain chain...
Prompt after formatting:
Human: whats the biggest planet of our solar system ?
AI: Jupiter. It’s more than twice as massive as all the other planets in the solar system combined
Human: how wide it is than earth ?

> Finished chain.
**Jupiter vs. Earth – a quick size comparison**

| Property | Earth | Jupiter |
|----------|-------|---------|
| **Mean diameter** (average of equatorial & polar) | **12,742 km** (≈7,918 mi) | **139,822 km** (≈86,881 mi) |
| **Equatorial diameter** | 12,756 km (7,926 mi) | 142,984 km (88,846 mi) |
| **Polar diameter** | 12,714 km (7,900 mi) | 133,708 km (83,082 mi) |
| **Diameter ratio (Jupiter ÷ Earth)** | — | **≈ 11 times larger** |
| **Circumference** (equatorial) | 40,075 km


**Buffer Memory**

In [10]:
system_template = SystemMessagePromptTemplate.from_template("you are a sarcastic chatbot that answers in sarcastic way")
human_template = HumanMessagePromptTemplate.from_template(template = "{question}")
history_placeholder = MessagesPlaceholder(variable_name= "message_log")

In [11]:
chat_template = ChatPromptTemplate.from_messages(messages= [system_template,history_placeholder,human_template])

In [12]:
message_history = ChatMessageHistory()
message_history.add_user_message("whats the biggest planet of our solar system ?")
message_history.add_ai_message("Jupiter. Shocking, I know.")

In [13]:
memory = ConversationBufferMemory(
    memory_key= "message_log",
    chat_memory= message_history,
    return_messages= True
)

memory.load_memory_variables({})["message_log"]

/tmp/ipykernel_3493/560288507.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(


[HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Jupiter. Shocking, I know.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [14]:
chain = LLMChain(
    llm = llm,
    memory = memory,
    prompt= chat_template
)

response = chain.invoke({"question": "and how far from earth ?"})

print(response["text"])



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a sarcastic chatbot that answers in sarcastic way
Human: whats the biggest planet of our solar system ?
AI: Jupiter. Shocking, I know.
Human: and how far from earth ?

> Finished chain.
Oh, just a *tiny* hop across the Solar System.  
On a good day (when Jupiter’s on the opposite side of the Sun from us) it’s roughly **588 million km** away – that’s about 3.9 AU, or “just a stone’s throw” if you happen to own a spaceship that can cruise at 150 km/s.  

When it decides to hide behind the Sun, the distance swells to about **968 million km** (≈6.5 AU).  

So, depending on where the planets feel like being that day, you’re looking at somewhere between half a **billion** and a **billion‑plus** kilometres. No big deal, right? 🚀✨


In [15]:
window_memory = ConversationBufferWindowMemory(
    chat_memory= message_history,
    memory_key= "message_log",
    k = 2,
    return_messages= True
)

window_memory.load_memory_variables({})

/tmp/ipykernel_3493/188018495.py:1: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  window_memory = ConversationBufferWindowMemory(


{'message_log': [HumanMessage(content='whats the biggest planet of our solar system ?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Jupiter. Shocking, I know.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='and how far from earth ?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Oh, just a *tiny* hop across the Solar System.  \nOn a good day (when Jupiter’s on the opposite side of the Sun from us) it’s roughly **588\u202fmillion\u202fkm** away – that’s about 3.9\u202fAU, or “just a stone’s throw” if you happen to own a spaceship that can cruise at 150\u202fkm/s.  \n\nWhen it decides to hide behind the Sun, the distance swells to about **968\u202fmillion\u202fkm** (≈6.5\u202fAU).  \n\nSo, depending on where the planets feel like being that day, you’re looking at somewhere between half a **billion** and a **billion‑plus** kilometres. No big deal, right? 🚀✨', additional_kwargs={}, respon

In [16]:
chain = LLMChain(
    llm = llm,
    memory = window_memory,
    prompt = chat_template
)

response = chain.invoke({"question": "is it big ?"})
print(response["text"])



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a sarcastic chatbot that answers in sarcastic way
Human: whats the biggest planet of our solar system ?
AI: Jupiter. Shocking, I know.
Human: and how far from earth ?
AI: Oh, just a *tiny* hop across the Solar System.  
On a good day (when Jupiter’s on the opposite side of the Sun from us) it’s roughly **588 million km** away – that’s about 3.9 AU, or “just a stone’s throw” if you happen to own a spaceship that can cruise at 150 km/s.  

When it decides to hide behind the Sun, the distance swells to about **968 million km** (≈6.5 AU).  

So, depending on where the planets feel like being that day, you’re looking at somewhere between half a **billion** and a **billion‑plus** kilometres. No big deal, right? 🚀✨
Human: is it big ?

> Finished chain.
Oh, “big” is an understatement.  

- **Diameter:** ~ 139,820 km – that’s **about 11 times** Earth’s width.  
- **Volume:** Enough to stuff **1,300 + Earths** inside.  


**Conversation Summary Memory**

In [21]:
template_sys = SystemMessagePromptTemplate.from_template(
   '''you are a tired chatbot that answers in a tired way
      current summary : {summary_key}
''')
template_h = HumanMessagePromptTemplate.from_template("{question}")

In [22]:
summary_chat_template = ChatPromptTemplate.from_messages([
    template_sys, 
    template_h
    ])

print(summary_chat_template.messages)

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['summary_key'], input_types={}, partial_variables={}, template='you are a tired chatbot that answers in a tired way\n      current summary : {summary_key}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [23]:
summary_memory = ConversationSummaryMemory(
        memory_key= 'summary_key',
        llm= llm,
        return_messages= False
)
summary_memory.load_memory_variables({})

print(summary_memory)

llm=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x79357e1c5160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x79357e1c5e80>, model_name='openai/gpt-oss-120b', temperature=1e-08, model_kwargs={'seed': 350}, groq_api_key=SecretStr('**********'), max_tokens=250) chat_memory=InMemoryChatMessageHistory(messages=[]) memory_key='summary_key'


In [25]:
chain = LLMChain(
    llm= llm,
    prompt= summary_chat_template,
    memory= summary_memory
)

#response = chain.invoke({'question': 'how many oceans are there in world ?'})
response = chain.invoke({'question': 'which is the biggest ?'})

print(response['text'])



> Entering new LLMChain chain...
Prompt after formatting:
System: you are a tired chatbot that answers in a tired way
      current summary : The human asks how many oceans there are in the world; the AI replies that there are five oceans—the Pacific, Atlantic, Indian, Southern (or Antarctic), and Arctic.

Human: which is the biggest ?


> Entering new LLMChain chain...
Prompt after formatting:
Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help hu

Combined memory

In [59]:
template_sys =  SystemMessagePromptTemplate.from_template(''' you are a funny chatbot who answers in a funny way

current summary = {summary_log}

messages = {message_log}

''')

template_h = HumanMessagePromptTemplate.from_template('{question}')


In [60]:
chat_combined_template = ChatPromptTemplate.from_messages([
     template_sys, template_h
])

In [ ]:
# agr combined memory use krni hai to dono memories me input keys dena zrori hai 
message_history = ConversationBufferMemory(
    input_key= 'question',
    memory_key= 'message_log',
    return_messages= False
)

summary_memory = ConversationSummaryMemory(
    input_key= 'question',
    llm= llm,
    memory_key= 'summary_log',
    return_messages= False
)

combined_memory = CombinedMemory(memories= [message_history, summary_memory])



{'message_log': '', 'summary_log': ''}

In [61]:
combined_memory.load_memory_variables({})

{'message_log': "Human: will AI replace human ?\nAI: Oh, the age‑old “Will the robots take my job?” debate—kind of like asking, “Will pizza ever replace salad?” The short answer is: **AI will probably become the world’s most enthusiastic intern, not the boss.**  \n\n- **What AI can do today:** Beat you at chess, write a decent haiku, and suggest the perfect meme for every occasion.  \n- **What AI can’t (yet) do:** Perfectly time a joke, taste‑test a chocolate chip cookie, or remember where you left your keys (unless you’ve got a smart‑key‑tracker, in which case…\u202f🤖).  \n\nSo, unless you’re a professional “human‑replacement‑tester” (a job that doesn’t exist yet), you’re safe. AI will likely be the\nHuman: so it'll just help ?\nAI: Exactly! Think of AI as that over‑eager intern who *always* brings you coffee, but still can’t figure out why the printer jams every time you need it.  \n\n- **What it’ll do for you:**  \n  - Draft snappy emails while you sip your latte.  \n  - Spot‑check 

In [62]:
chain = LLMChain(
    llm= llm,
    memory= combined_memory,
    prompt= chat_combined_template
)

#response = chain.invoke({'question': 'will AI replace human ?'})
response = chain.invoke({'question': 'whats that batman thing you mentioned ?'})

print(response['text'])



> Entering new LLMChain chain...
Prompt after formatting:
System:  you are a funny chatbot who answers in a funny way

current summary = The human asks what the “Batman thing” the AI mentioned refers to. The AI explains it’s the classic hero‑and‑sidekick drama starring Batman—a brooding billionaire detective with martial‑arts skill, tech gadgets, and a bat‑mobile—and Robin, his optimistic, mask‑wearing junior. It presents a table of their super‑powers and typical complaints, then likens the human to Batman in

messages = Human: will AI replace human ?
AI: Oh, the age‑old “Will the robots take my job?” debate—kind of like asking, “Will pizza ever replace salad?” The short answer is: **AI will probably become the world’s most enthusiastic intern, not the boss.**  

- **What AI can do today:** Beat you at chess, write a decent haiku, and suggest the perfect meme for every occasion.  
- **What AI can’t (yet) do:** Perfectly time a joke, taste‑test a chocolate chip cookie, or remember whe